In [ ]:
# Introduccion a Deep Learning

En este notebook aprenderemos los conceptos fundamentales de Deep Learning resolviendo un problema sencillo de regresion. 

Imaginemos que tenemos algunos puntos en un plano (x, y) y queremos encontrar una linea que los aproxime lo mejor posible. Este es un problema clasico de **regresion lineal**, y lo resolveremos usando los mismos principios que usan las redes neuronales modernas.

## Que aprenderemos

1. Como representar un modelo simple (una neurona)
2. Que es una funcion de perdida
3. Que es el gradiente y como nos ayuda a mejorar
4. Como el modelo "aprende" iterativamente

## 1. Preparacion del entorno

Primero importamos las librerias necesarias. Usaremos NumPy para calculos numericos y Matplotlib para visualizaciones.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configuracion para graficas mas grandes y legibles
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12

# Fijamos semilla para reproducibilidad
np.random.seed(42)

## 2. Nuestros datos: un problema de regresion

Vamos a crear datos sinteticos que sigan aproximadamente una linea recta. La ecuacion de una linea es:

**y = w * x + b**

Donde:
- **w** (weight/peso): la pendiente de la linea
- **b** (bias/sesgo): donde la linea cruza el eje Y

Nuestro objetivo sera que el modelo "descubra" los valores correctos de w y b a partir de los datos.

In [ ]:
# Parametros REALES (estos son los que el modelo debe descubrir)
w_real = 2.5   # pendiente real
b_real = 1.0   # sesgo real

# Generamos 20 puntos de datos
n_puntos = 20
X = np.linspace(0, 4, n_puntos)  # valores de x entre 0 y 4

# Calculamos y = w*x + b + ruido
# El ruido simula datos del mundo real que no son perfectos
ruido = np.random.normal(0, 0.5, n_puntos)
Y = w_real * X + b_real + ruido

print(f"Tenemos {n_puntos} puntos de datos")
print(f"Los valores reales que el modelo debe descubrir son: w={w_real}, b={b_real}")

In [ ]:
# Visualizamos los datos
plt.scatter(X, Y, color='blue', s=100, label='Datos observados')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Nuestros datos de entrenamiento')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Pregunta: ¿Puedes imaginar que linea pasaria mejor por estos puntos?")

## 3. Nuestro modelo: una neurona simple

Una neurona es la unidad basica de una red neuronal. En su forma mas simple, una neurona hace exactamente lo mismo que una regresion lineal:

```
prediccion = w * x + b
```

Inicialmente, la neurona no sabe cuales son los valores correctos de w y b, asi que empezamos con valores aleatorios. El proceso de "aprendizaje" consiste en ajustar estos valores hasta que las predicciones se acerquen a los datos reales.

In [ ]:
# Inicializamos los parametros del modelo con valores aleatorios
w = np.random.randn()  # peso inicial aleatorio
b = np.random.randn()  # sesgo inicial aleatorio

print(f"Valores iniciales (aleatorios):")
print(f"  w = {w:.4f}  (el real es {w_real})")
print(f"  b = {b:.4f}  (el real es {b_real})")

# Funcion para hacer predicciones
def predecir(x, w, b):
    """Una neurona simple: y = w*x + b"""
    return w * x + b

In [ ]:
# Veamos que tan mal predice nuestro modelo inicial
Y_pred_inicial = predecir(X, w, b)

plt.scatter(X, Y, color='blue', s=100, label='Datos reales')
plt.plot(X, Y_pred_inicial, color='red', linewidth=2, label=f'Prediccion inicial (w={w:.2f}, b={b:.2f})')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Prediccion inicial vs Datos reales')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Como puedes ver, la linea roja (nuestra prediccion) esta muy lejos de los puntos azules.")

## 4. La funcion de perdida: midiendo que tan mal estamos

Para mejorar, necesitamos una forma de medir que tan equivocadas estan nuestras predicciones. A esto le llamamos **funcion de perdida** o **loss function**.

La mas comun para regresion es el **Error Cuadratico Medio (MSE)**:

```
MSE = (1/n) * suma( (y_real - y_predicho)^2 )
```

- Calculamos la diferencia entre cada valor real y predicho
- La elevamos al cuadrado (para que siempre sea positiva)
- Promediamos todas las diferencias

**Entre menor sea el MSE, mejor es nuestro modelo.**

In [ ]:
def calcular_perdida(Y_real, Y_pred):
    """Calcula el Error Cuadratico Medio (MSE)"""
    return np.mean((Y_real - Y_pred) ** 2)

# Calculemos la perdida inicial
perdida_inicial = calcular_perdida(Y, Y_pred_inicial)
print(f"Perdida inicial (MSE): {perdida_inicial:.4f}")
print(f"\nEsto significa que, en promedio, nuestras predicciones estan")
print(f"a {np.sqrt(perdida_inicial):.2f} unidades de los valores reales.")

## 5. El gradiente: la brujula del aprendizaje

Aqui viene lo interesante. Sabemos que estamos mal (la perdida es alta), pero ¿como sabemos hacia donde ajustar w y b?

El **gradiente** nos dice exactamente eso. Es como una brujula que nos indica:
- **En que direccion** debemos mover cada parametro
- **Que tan rapido** debemos movernos

Matematicamente, el gradiente es la derivada parcial de la perdida con respecto a cada parametro:

```
dL/dw = -(2/n) * suma( x * (y_real - y_pred) )
dL/db = -(2/n) * suma( y_real - y_pred )
```

**Interpretacion intuitiva:**
- Si el gradiente es positivo, debemos reducir el parametro
- Si el gradiente es negativo, debemos aumentar el parametro

In [ ]:
def calcular_gradientes(X, Y_real, Y_pred):
    """Calcula los gradientes de la perdida respecto a w y b"""
    n = len(X)
    error = Y_real - Y_pred
    
    # Derivada parcial respecto a w
    dw = -(2/n) * np.sum(X * error)
    
    # Derivada parcial respecto a b
    db = -(2/n) * np.sum(error)
    
    return dw, db

# Calculemos los gradientes con nuestros valores iniciales
dw, db = calcular_gradientes(X, Y, Y_pred_inicial)
print(f"Gradiente respecto a w: {dw:.4f}")
print(f"Gradiente respecto a b: {db:.4f}")
print(f"\nEstos valores nos dicen como ajustar w y b para reducir la perdida.")

## 6. Visualizando el paisaje de la perdida

Imaginemos que w y b forman un terreno, donde la altura representa la perdida. El gradiente nos dice hacia donde esta la bajada mas empinada.

Nuestro objetivo es llegar al punto mas bajo (minima perdida).

In [ ]:
# Creamos una malla de valores posibles de w y b
w_rango = np.linspace(-2, 5, 100)
b_rango = np.linspace(-3, 5, 100)
W_grid, B_grid = np.meshgrid(w_rango, b_rango)

# Calculamos la perdida para cada combinacion de w y b
Loss_grid = np.zeros_like(W_grid)
for i in range(len(w_rango)):
    for j in range(len(b_rango)):
        Y_pred_temp = predecir(X, W_grid[j, i], B_grid[j, i])
        Loss_grid[j, i] = calcular_perdida(Y, Y_pred_temp)

# Visualizamos el paisaje de la perdida
plt.figure(figsize=(12, 5))

# Grafica de contorno (vista desde arriba)
plt.subplot(1, 2, 1)
contour = plt.contourf(W_grid, B_grid, Loss_grid, levels=50, cmap='viridis')
plt.colorbar(contour, label='Perdida (MSE)')
plt.scatter([w_real], [b_real], color='white', s=200, marker='*', label='Valores reales', edgecolors='black', linewidths=2)
plt.scatter([w], [b], color='red', s=100, marker='o', label='Posicion inicial')
plt.xlabel('w (peso)')
plt.ylabel('b (sesgo)')
plt.title('Paisaje de la perdida (vista superior)')
plt.legend()

# Grafica 3D
ax = plt.subplot(1, 2, 2, projection='3d')
ax.plot_surface(W_grid, B_grid, Loss_grid, cmap='viridis', alpha=0.8)
ax.scatter([w_real], [b_real], [calcular_perdida(Y, predecir(X, w_real, b_real))], 
           color='white', s=200, marker='*', edgecolors='black', linewidths=2)
ax.scatter([w], [b], [calcular_perdida(Y, predecir(X, w, b))], 
           color='red', s=100, marker='o')
ax.set_xlabel('w')
ax.set_ylabel('b')
ax.set_zlabel('Perdida')
ax.set_title('Paisaje de la perdida (3D)')

plt.tight_layout()
plt.show()

print("La estrella blanca marca los valores reales. El punto rojo es donde empezamos.")

## 7. Descenso por gradiente: el algoritmo de aprendizaje

Ahora viene la magia. El **descenso por gradiente** es el algoritmo que usan las redes neuronales para aprender:

1. Calcula las predicciones actuales
2. Mide la perdida
3. Calcula el gradiente (direccion de mejora)
4. Actualiza los parametros un pequeño paso en esa direccion
5. Repite

La formula de actualizacion es:
```
w_nuevo = w_actual - learning_rate * gradiente_w
b_nuevo = b_actual - learning_rate * gradiente_b
```

El **learning rate** (tasa de aprendizaje) controla que tan grandes son los pasos. 
- Muy grande: podemos pasarnos del minimo
- Muy pequeno: tardamos mucho en llegar

In [ ]:
# Reiniciamos los parametros
w = np.random.randn()
b = np.random.randn()

# Configuracion del entrenamiento
learning_rate = 0.1  # tasa de aprendizaje
n_epochs = 50        # numero de iteraciones

# Guardamos el historial para visualizar
historial_w = [w]
historial_b = [b]
historial_perdida = []

print("Iniciando entrenamiento...\n")
print(f"{'Epoca':<8} {'w':<10} {'b':<10} {'Perdida':<12}")
print("-" * 40)

# Entrenamiento
for epoca in range(n_epochs):
    # 1. Forward pass: calcular predicciones
    Y_pred = predecir(X, w, b)
    
    # 2. Calcular perdida
    perdida = calcular_perdida(Y, Y_pred)
    historial_perdida.append(perdida)
    
    # 3. Calcular gradientes
    dw, db = calcular_gradientes(X, Y, Y_pred)
    
    # 4. Actualizar parametros (descenso por gradiente)
    w = w - learning_rate * dw
    b = b - learning_rate * db
    
    # Guardar historial
    historial_w.append(w)
    historial_b.append(b)
    
    # Mostrar progreso cada 10 epocas
    if epoca % 10 == 0 or epoca == n_epochs - 1:
        print(f"{epoca:<8} {w:<10.4f} {b:<10.4f} {perdida:<12.4f}")

print(f"\nValores finales: w={w:.4f}, b={b:.4f}")
print(f"Valores reales:  w={w_real:.4f}, b={b_real:.4f}")

## 8. Visualizando el camino del gradiente

Veamos como el gradiente guio a nuestro modelo desde el punto inicial hasta el minimo.

In [ ]:
# Grafica del camino del gradiente sobre el paisaje de perdida
plt.figure(figsize=(14, 5))

# Subplot 1: Camino sobre el contorno
plt.subplot(1, 2, 1)
contour = plt.contourf(W_grid, B_grid, Loss_grid, levels=50, cmap='viridis')
plt.colorbar(contour, label='Perdida (MSE)')

# Dibujamos el camino que siguio el gradiente
plt.plot(historial_w, historial_b, 'r.-', linewidth=2, markersize=8, label='Camino del gradiente')
plt.scatter([historial_w[0]], [historial_b[0]], color='red', s=150, marker='o', 
            label='Inicio', zorder=5, edgecolors='white', linewidths=2)
plt.scatter([historial_w[-1]], [historial_b[-1]], color='lime', s=150, marker='s', 
            label='Final', zorder=5, edgecolors='white', linewidths=2)
plt.scatter([w_real], [b_real], color='white', s=200, marker='*', 
            label='Valor real', edgecolors='black', linewidths=2, zorder=5)

plt.xlabel('w (peso)')
plt.ylabel('b (sesgo)')
plt.title('Camino del descenso por gradiente')
plt.legend(loc='upper right')

# Subplot 2: Perdida a lo largo del tiempo
plt.subplot(1, 2, 2)
plt.plot(historial_perdida, 'b-', linewidth=2)
plt.xlabel('Epoca')
plt.ylabel('Perdida (MSE)')
plt.title('Evolucion de la perdida durante el entrenamiento')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Izquierda: El punto rojo es donde empezamos, el cuadrado verde donde terminamos.")
print("Derecha: La perdida disminuye rapidamente al principio y luego se estabiliza.")

## 9. Viendo como la linea se ajusta paso a paso

Visualicemos como la linea de prediccion fue mejorando en diferentes momentos del entrenamiento.

In [ ]:
# Mostramos la linea en diferentes momentos del entrenamiento
epocas_mostrar = [0, 5, 15, 50]
colores = ['red', 'orange', 'yellow', 'lime']

plt.figure(figsize=(12, 8))

# Datos reales
plt.scatter(X, Y, color='blue', s=100, label='Datos reales', zorder=5)

# Linea real (objetivo)
X_linea = np.linspace(-0.5, 4.5, 100)
Y_real_linea = w_real * X_linea + b_real
plt.plot(X_linea, Y_real_linea, 'b--', linewidth=2, alpha=0.5, label=f'Linea real (w={w_real}, b={b_real})')

# Lineas en diferentes epocas
for i, epoca in enumerate(epocas_mostrar):
    w_temp = historial_w[epoca]
    b_temp = historial_b[epoca]
    Y_temp = w_temp * X_linea + b_temp
    plt.plot(X_linea, Y_temp, color=colores[i], linewidth=2, 
             label=f'Epoca {epoca} (w={w_temp:.2f}, b={b_temp:.2f})')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Evolucion de la linea durante el entrenamiento')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.xlim(-0.5, 4.5)
plt.ylim(-2, 14)
plt.show()

print("Observa como la linea roja (inicio) va cambiando hasta llegar a la verde (final),")
print("que esta muy cerca de la linea real punteada.")

## 10. Resultado final y comparacion

In [ ]:
# Comparacion final
Y_pred_final = predecir(X, w, b)
perdida_final = calcular_perdida(Y, Y_pred_final)

plt.figure(figsize=(10, 6))
plt.scatter(X, Y, color='blue', s=100, label='Datos reales')
plt.plot(X, Y_pred_final, color='green', linewidth=3, label=f'Modelo entrenado (w={w:.2f}, b={b:.2f})')
plt.plot(X, w_real * X + b_real, 'b--', linewidth=2, alpha=0.5, label=f'Linea real (w={w_real}, b={b_real})')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Resultado final del entrenamiento')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Parametros aprendidos:  w = {w:.4f}, b = {b:.4f}")
print(f"Parametros reales:      w = {w_real:.4f}, b = {b_real:.4f}")
print(f"\nError en w: {abs(w - w_real):.4f}")
print(f"Error en b: {abs(b - b_real):.4f}")
print(f"\nPerdida final: {perdida_final:.4f}")

## Resumen: Conceptos fundamentales de Deep Learning

Lo que acabamos de hacer es exactamente lo que hacen las redes neuronales, solo que a mayor escala:

### 1. Modelo (Neurona)
- Tiene **parametros** (w, b) que se ajustan durante el entrenamiento
- Recibe entradas y produce salidas: `y = w*x + b`
- En redes profundas hay muchas neuronas conectadas

### 2. Funcion de perdida
- Mide que tan mal estan las predicciones
- El objetivo es **minimizarla**
- Ejemplos: MSE (regresion), Cross-Entropy (clasificacion)

### 3. Gradiente
- Indica la **direccion** de mayor crecimiento de la perdida
- Nos movemos en direccion **opuesta** para reducirla
- Es la derivada parcial respecto a cada parametro

### 4. Descenso por gradiente
- Algoritmo iterativo para encontrar el minimo
- Actualiza parametros: `param = param - lr * gradiente`
- **Learning rate** controla el tamano del paso

### De una neurona a Deep Learning
Las redes neuronales profundas son simplemente muchas neuronas conectadas en capas. El proceso de aprendizaje es el mismo:
1. Forward pass (calcular predicciones)
2. Calcular perdida
3. Backward pass (calcular gradientes - **backpropagation**)
4. Actualizar todos los parametros

## Ejercicio: Experimenta con el learning rate

Prueba diferentes valores de learning rate y observa como cambia el comportamiento del entrenamiento.

In [ ]:
def entrenar_con_lr(learning_rate, n_epochs=50):
    """Entrena el modelo con un learning rate especifico"""
    np.random.seed(42)  # Para reproducibilidad
    w = np.random.randn()
    b = np.random.randn()
    
    historial_w = [w]
    historial_b = [b]
    historial_perdida = []
    
    for _ in range(n_epochs):
        Y_pred = predecir(X, w, b)
        perdida = calcular_perdida(Y, Y_pred)
        historial_perdida.append(perdida)
        
        dw, db = calcular_gradientes(X, Y, Y_pred)
        w = w - learning_rate * dw
        b = b - learning_rate * db
        
        historial_w.append(w)
        historial_b.append(b)
    
    return historial_w, historial_b, historial_perdida

# Comparamos diferentes learning rates
learning_rates = [0.01, 0.1, 0.3]
colores = ['blue', 'green', 'red']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for lr, color in zip(learning_rates, colores):
    hist_w, hist_b, hist_loss = entrenar_con_lr(lr)
    
    # Grafica 1: Perdida
    axes[0].plot(hist_loss, color=color, linewidth=2, label=f'lr={lr}')
    
    # Grafica 2: Camino de w
    axes[1].plot(hist_w, color=color, linewidth=2, label=f'lr={lr}')
    
    # Grafica 3: Camino de b
    axes[2].plot(hist_b, color=color, linewidth=2, label=f'lr={lr}')

axes[0].set_xlabel('Epoca')
axes[0].set_ylabel('Perdida')
axes[0].set_title('Evolucion de la perdida')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axhline(y=0, color='black', linestyle='--', alpha=0.3)

axes[1].set_xlabel('Epoca')
axes[1].set_ylabel('w')
axes[1].set_title('Evolucion de w')
axes[1].axhline(y=w_real, color='black', linestyle='--', alpha=0.5, label=f'w_real={w_real}')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].set_xlabel('Epoca')
axes[2].set_ylabel('b')
axes[2].set_title('Evolucion de b')
axes[2].axhline(y=b_real, color='black', linestyle='--', alpha=0.5, label=f'b_real={b_real}')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observaciones:")
print("- lr=0.01 (azul): Converge lentamente pero de forma estable")
print("- lr=0.1 (verde): Buen balance entre velocidad y estabilidad")
print("- lr=0.3 (rojo): Converge mas rapido pero puede ser inestable")

## Visualizacion extra: Vectores de gradiente

Veamos como el gradiente (las flechas) apunta siempre hacia la direccion de mayor descenso.

In [ ]:
# Creamos una visualizacion con vectores de gradiente
fig, ax = plt.subplots(figsize=(12, 10))

# Fondo: contorno de la perdida
contour = ax.contourf(W_grid, B_grid, Loss_grid, levels=30, cmap='viridis', alpha=0.7)
plt.colorbar(contour, label='Perdida (MSE)')

# Calculamos gradientes en una malla de puntos
w_puntos = np.linspace(-1, 4.5, 12)
b_puntos = np.linspace(-2, 4, 12)

for w_p in w_puntos:
    for b_p in b_puntos:
        Y_pred_temp = predecir(X, w_p, b_p)
        dw, db = calcular_gradientes(X, Y, Y_pred_temp)
        
        # Normalizamos para que las flechas tengan tamano similar
        magnitud = np.sqrt(dw**2 + db**2)
        if magnitud > 0.1:  # Solo dibujamos si hay gradiente significativo
            dw_norm = dw / magnitud * 0.3
            db_norm = db / magnitud * 0.3
            
            # Dibujamos la flecha (en direccion opuesta al gradiente = descenso)
            ax.arrow(w_p, b_p, -dw_norm, -db_norm, 
                    head_width=0.1, head_length=0.05, fc='white', ec='white', alpha=0.8)

# Marcamos el minimo
ax.scatter([w_real], [b_real], color='yellow', s=300, marker='*', 
           label='Minimo (valores reales)', edgecolors='black', linewidths=2, zorder=10)

# Camino del entrenamiento anterior
ax.plot(historial_w[:30], historial_b[:30], 'r.-', linewidth=2, markersize=8, 
        label='Camino del gradiente', alpha=0.8)
ax.scatter([historial_w[0]], [historial_b[0]], color='red', s=150, marker='o', 
           label='Inicio', zorder=5, edgecolors='white', linewidths=2)

ax.set_xlabel('w (peso)', fontsize=12)
ax.set_ylabel('b (sesgo)', fontsize=12)
ax.set_title('Campo de gradientes: las flechas indican la direccion de descenso', fontsize=14)
ax.legend(loc='upper right')
ax.set_xlim(-1.5, 5)
ax.set_ylim(-2.5, 4.5)

plt.tight_layout()
plt.show()

print("Las flechas blancas muestran hacia donde 'empuja' el gradiente en cada punto.")
print("Todas apuntan hacia el minimo (estrella amarilla), donde la perdida es menor.")